# 03 — Feature engineering

Enriches the region split tables with all static and dynamic features needed by the LGB model.

Applied to both `region_split` (3-timestamp) and `region_inference_only` (2-timestamp).
Inference-only locations receive the same feature schema; missing values are imputed where justified.

**Feature groups:**
1. Vegetation class — dominant by polygon area (WFS vegetatielegger)
2. Land use — dominant by polygon area (BRP gewas)
3. Soil group — dominant intersection with BRO Bodemkaart (~30s spatial overlay)
4. Nearest discharge station — spatial nearest-neighbour join
5. HW window metrics — n_events, max_rise_rate, drawdown_index, flood_days per t1↔t2 and t2↔t3 windows
6. Bend exposure — curvature features (loaded from pre-computed reference)
7. Ordinal encoding of all categorical columns

**Inputs** (`03_features/EXPERIMENT/`):
- `region_split.parquet` — 3-timestamp OK regions
- `region_inference_only.parquet` — 2-timestamp OK regions

**Outputs** (`03_features/EXPERIMENT/`):
- `region_features.parquet` — 29-column feature table (same schema as reference)
- `region_inference_features.parquet` — same schema, `v_test`/`split` omitted

**Control experiment** (section 9): asserts value-level parity with the reference
`02_processed/erosion/region_features.parquet` before proceeding to notebook 4.

In [ ]:
import os, sys
from pathlib import Path

_cwd = Path.cwd()
for candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (candidate / 'src').exists():
        _backend = candidate
        break
else:
    _backend = _cwd

os.chdir(_backend)
sys.path.insert(0, str(_backend))
print('cwd:', os.getcwd())

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd

import src.paths as PATHS
import src.constants as CONST

# ── Experiment config ─────────────────────────────────────────────────────────
EXPERIMENT   = '20260314'

FEATURES_DIR = PATHS.DATA_DIR / f'03_features/{EXPERIMENT}'
IN_SPLIT     = FEATURES_DIR / 'region_split.parquet'
IN_INFERENCE = FEATURES_DIR / 'region_inference_only.parquet'
OUT_FEATURES = FEATURES_DIR / 'region_features.parquet'
OUT_INFERENCE_FEATURES = FEATURES_DIR / 'region_inference_features.parquet'

# Source data paths
PROC_GPKG    = PATHS.DATA_DIR / '02_processed/erosion/wocu_post_processed_fase2_20260310.gpkg'
SCOPE_GPKG   = PATHS.DATA_DIR / '01_raw/scope/scope_fase2.gpkg'
VEG_GPKG     = PATHS.DATA_DIR / '02_processed/wfs_context/vegetatielegger.gpkg'
LU_GPKG      = PATHS.DATA_DIR / '02_processed/wfs_context/land_use.gpkg'
SOIL_GPKG    = PATHS.DATA_DIR / '01_raw/soil/BRO_DownloadBodemkaart.gpkg'
STATIONS_GPKG = PATHS.DATA_DIR / '02_processed/water_stations/water_stations_for_modeling.gpkg'
DISCHARGE_DIR = PATHS.DATA_DIR / 'timeseries/discharge'

# Reference parquet for parity check (end of notebook)
REFERENCE_FEATURES = PATHS.DATA_DIR / '02_processed/erosion/region_features.parquet'
REFERENCE_FEATURES_V2 = PATHS.DATA_DIR / '02_processed/erosion/region_features_v2.parquet'

# High-water threshold per discharge station (m³/s) — decided in explore_features.ipynb
HIGH_WATER_THRESHOLD = {
    'lobith.bovenrijn.tolkamer': 6000,
    'millingenaanderijn': 4000,
    'pannerden.pannerdenschkanaal': 2000,
    'driel.boven': 1000,
    'hagestein.boven': 1000,
    'maastricht.borgharen.maas.beneden': 1000,
    'maaseik': 1000,
    'venlo': 1000,
    'megen.maas': 1000,
    'hank.bergschemaas': 1100,
    'westervoort.ijsselkop': 800,
    'westervoort': 800,
    'olst': 800,
    'genemuiden': 300,
}

DT_HOURS  = 1 / 6   # 10-minute sampling interval in hours
GAP_HOURS = 72       # merge HW events separated by < 72h

def encode_col(series: pd.Series, category_key: str) -> pd.Series:
    mapping = {v: i for i, v in enumerate(CONST.KNOWN_CATEGORIES[category_key])}
    return series.map(lambda x: mapping.get(x, CONST.DEFAULT_UNKNOWN_CATEGORY_LABEL))

def soil_group_fn(code):
    if pd.isna(code): return np.nan
    c = str(code)
    if c[:2] in ('Rd', 'Rn', 'Ro'):              return 'River clay'
    elif c[0] == 'Z':                              return 'Sandy'
    elif c[:2] in ('Mv', 'Mo', 'pM') or c[0] == 'b': return 'Soft (peat/podzol)'
    elif c[:2] == 'Mn':                            return 'River sand-clay'
    else:                                          return 'Other'

print(f'Experiment : {EXPERIMENT}')

## 1. Load region splits

In [ ]:
split      = pd.read_parquet(IN_SPLIT)
inference  = pd.read_parquet(IN_INFERENCE)

# Add river column for encoding
split['river']     = split.index.to_series().str.extract(r'^([a-z]+\d*)_')[0]
inference['river'] = inference.index.to_series().str.extract(r'^([a-z]+\d*)_')[0]

print(f'region_split        : {split.shape}')
print(f'region_inference    : {inference.shape}')

# Combined index for shared spatial joins
all_ids = split.index.union(inference.index)
print(f'Combined locations  : {len(all_ids):,}')

## 2. Load scope geometries (used in multiple joins below)

In [ ]:
print('Loading scope geometries ...')
# The old GPKG (fase2) uses vlakken_scope with position_id; use it for spatial joins
scope_raw = gpd.read_file(SCOPE_GPKG, layer='vlakken_scope')
if 'position_id' in scope_raw.columns and 'location_id' not in scope_raw.columns:
    scope_raw = scope_raw.rename(columns={'position_id': 'location_id'})
scope_raw = scope_raw.set_index('location_id')[['geometry']]
# Keep only locations in our combined set
scope = scope_raw.loc[scope_raw.index.intersection(all_ids)]
print(f'  Scope geometries loaded: {len(scope):,}')

## 3. Vegetation class

In [ ]:
print('Loading vegetation (dominant class by area) ...')
veg = gpd.read_file(VEG_GPKG, layer='rws_vegetatielegger:vegetatieklassen')
veg['area'] = veg.geometry.area
dom_veg = (
    veg.sort_values('area', ascending=False)
    .groupby('scope_region_id')['vlklasse']
    .first()
    .rename('vegetation_class')
)
# Consolidate rare mix classes to 'Other'
rare_classes = {'90/10', '70/30', '50/50'}
dom_veg = dom_veg.apply(lambda x: 'Other' if x in rare_classes else x)

split['vegetation_class']     = split.index.map(dom_veg)
inference['vegetation_class'] = inference.index.map(dom_veg)
print(f'  Assigned: split={split["vegetation_class"].notna().sum():,}  '
      f'inference={inference["vegetation_class"].notna().sum():,}')
print(f'  Distribution: {split["vegetation_class"].value_counts().to_dict()}')

## 4. Land use

In [ ]:
print('Loading land use (dominant class by area) ...')
lu = gpd.read_file(LU_GPKG, layer='BrpGewas')
lu['area'] = lu.geometry.area
dom_lu = (
    lu.sort_values('area', ascending=False)
    .groupby('scope_region_id')['category']
    .first()
    .rename('land_use')
)
split['land_use']     = split.index.map(dom_lu)
inference['land_use'] = inference.index.map(dom_lu)
print(f'  Assigned: split={split["land_use"].notna().sum():,}  '
      f'inference={inference["land_use"].notna().sum():,}')

## 5. Soil group  (spatial overlay — ~30s)

In [ ]:
print('Loading Bodemkaart (spatial overlay, ~30s) ...')
soil_poly  = gpd.read_file(SOIL_GPKG, layer='soilarea')[['maparea_id', 'geometry']]
soil_codes = gpd.read_file(SOIL_GPKG, layer='soilarea_soilunit')[['maparea_id', 'soilunit_code']]
soil_poly  = soil_poly.merge(soil_codes, on='maparea_id', how='left')

scope_for_soil = scope.reset_index()
joined_soil = gpd.overlay(
    scope_for_soil,
    soil_poly[['soilunit_code', 'geometry']],
    how='intersection', keep_geom_type=False,
)
joined_soil['area'] = joined_soil.geometry.area
dom_soil = (
    joined_soil.sort_values('area', ascending=False)
    .groupby('location_id')['soilunit_code']
    .first()
)
dom_soil_group = dom_soil.map(soil_group_fn)

split['soil_group']     = split.index.map(dom_soil_group)
inference['soil_group'] = inference.index.map(dom_soil_group)
print(f'  Assigned: split={split["soil_group"].notna().sum():,}  '
      f'inference={inference["soil_group"].notna().sum():,}')
print(f'  Distribution: {split["soil_group"].value_counts().to_dict()}')

## 6. Nearest discharge station

In [ ]:
print('Assigning nearest discharge station ...')
stations = gpd.read_file(STATIONS_GPKG, layer='discharge_stations')
stations = stations.rename(columns={'CODE': 'station_code'})
stations = stations.to_crs(scope.crs)

scope_centroid = scope.copy()
scope_centroid['geometry'] = scope_centroid.geometry.centroid

nearest = gpd.sjoin_nearest(
    scope_centroid.reset_index(),
    stations[['station_code', 'geometry']],
    how='left',
    distance_col='station_dist_m',
)
station_map = nearest.set_index('location_id')['station_code']

split['nearest_station']     = split.index.map(station_map)
inference['nearest_station'] = inference.index.map(station_map)
print(f'  Assigned: {split["nearest_station"].notna().sum():,} / {len(split):,} split regions')
print(f'  Distribution: {split["nearest_station"].value_counts().to_dict()}')

## 7. High-water window metrics

For each region, compute mean `n_events`, `max_rise_rate`, `drawdown_index`, `flood_days` over:
- `t1` window: years between t1 and t2 (exclusive)
- `t2` window: years between t2 and t3 (exclusive; only for 3-timestamp regions)

In [ ]:
print('Loading discharge timeseries (10-min resolution) ...')
_CLEANED = PATHS.DATA_DIR / 'water_stations_timeseries' / 'cleaned' / 'discharge'
_DISC_DIR = _CLEANED if _CLEANED.exists() and list(_CLEANED.glob('*.parquet')) else DISCHARGE_DIR
print(f'  Using: {_DISC_DIR}')

disc_10min = []
for f in sorted(_DISC_DIR.glob('*.parquet')):
    df_ = pd.read_parquet(f)
    df_['timestamp'] = pd.to_datetime(df_['timestamp'], utc=True)
    disc_10min.append(df_)
disc_raw = pd.concat(disc_10min, ignore_index=True)
print(f'  Records: {len(disc_raw):,}  |  Stations: {disc_raw["station_code"].unique().tolist()}')

In [ ]:
def compute_high_water_metrics(station_code: str, df: pd.DataFrame, q_thresh: float) -> pd.DataFrame:
    """Compute n_events, max_rise_rate, drawdown_index per calendar year for one station."""
    df = df.sort_values('timestamp').copy()
    df['year']    = df['timestamp'].dt.year
    df['exceed']  = df['discharge_m3s'] > q_thresh
    df['excess']  = np.maximum(df['discharge_m3s'] - q_thresh, 0)
    df['dQ']      = df['discharge_m3s'].diff()
    df['dt_h']    = df['timestamp'].diff().dt.total_seconds() / 3600
    df['dQ_dt']   = np.where(df['dt_h'] > 0, df['dQ'] / df['dt_h'], np.nan)

    rows = []
    for year, grp in df.groupby('year'):
        g = grp.dropna(subset=['discharge_m3s']).sort_values('timestamp')
        if g.empty:
            rows.append({'station_id': station_code, 'year': year,
                         'n_events': np.nan, 'max_rise_rate': np.nan,
                         'drawdown_index': np.nan, 'flood_days': np.nan})
            continue
        # Event detection with gap bridging
        exceed = g['exceed'].values
        ts = pd.to_datetime(g['timestamp'])
        blocks, i = [], 0
        while i < len(exceed):
            if exceed[i]:
                start = i
                while i < len(exceed) and exceed[i]: i += 1
                blocks.append((ts.iloc[start], ts.iloc[i - 1]))
            else:
                i += 1
        merged = []
        for t0, t1_ in blocks:
            if merged and (t0 - merged[-1][1]).total_seconds() / 3600 < GAP_HOURS:
                merged[-1] = (merged[-1][0], t1_)
            else:
                merged.append((t0, t1_))
        n_events = len(merged)
        rise_mask     = g['exceed'] & (g['dQ_dt'] > 0)
        recession_mask = g['exceed'] & (g['dQ_dt'] < 0)
        max_rise   = g.loc[rise_mask,     'dQ_dt'].max() if rise_mask.any()     else np.nan
        drawdown   = np.abs(g.loc[recession_mask, 'dQ_dt']).max() if recession_mask.any() else np.nan
        # flood_days: days with at least one 10-min reading above threshold
        flood_days = int(g.loc[g['exceed'], 'timestamp'].dt.date.nunique())
        rows.append({'station_id': station_code, 'year': year,
                     'n_events': n_events, 'max_rise_rate': max_rise,
                     'drawdown_index': drawdown, 'flood_days': flood_days})
    return pd.DataFrame(rows)

print('Computing high-water metrics per station ...')
hw_results = []
for code, thresh in HIGH_WATER_THRESHOLD.items():
    sub = disc_raw[disc_raw['station_code'] == code]
    if sub.empty:
        print(f'  WARNING: no data for station {code}')
        continue
    hw_results.append(compute_high_water_metrics(code, sub, thresh))
hw_metrics = pd.concat(hw_results, ignore_index=True).set_index(['station_id', 'year'])
print(f'HW metrics: {len(hw_metrics):,} (station × year) rows')

In [ ]:
HW_METRIC_COLS = ['n_events', 'max_rise_rate', 'drawdown_index', 'flood_days']

def hw_window_stats(row, t3_col=None):
    """Mean HW metrics over the t1↔t2 and (optionally) t2↔t3 windows."""
    code = row.get('nearest_station')
    t1, t2 = row.get('t1'), row.get('t2')
    t3 = row.get('t3') if t3_col else None
    nan_out = {f'{c}_t1': np.nan for c in HW_METRIC_COLS}
    nan_out.update({f'{c}_t2': np.nan for c in HW_METRIC_COLS})

    if pd.isna(code) or code not in hw_metrics.index.get_level_values(0):
        return pd.Series(nan_out)
    try:
        sub = hw_metrics.loc[code]
        sub = sub.to_frame().T if isinstance(sub, pd.Series) else sub
    except KeyError:
        return pd.Series(nan_out)

    out = {}
    years_t1 = list(range(int(t1) + 1, int(t2))) if not (pd.isna(t1) or pd.isna(t2)) and int(t2) - int(t1) > 1 else []
    years_t2 = list(range(int(t2) + 1, int(t3))) if t3 and not pd.isna(t3) and int(t3) - int(t2) > 1 else []

    for c in HW_METRIC_COLS:
        sub_t1 = sub.loc[sub.index.intersection(years_t1)] if years_t1 else pd.DataFrame()
        sub_t2 = sub.loc[sub.index.intersection(years_t2)] if years_t2 else pd.DataFrame()
        out[f'{c}_t1'] = sub_t1[c].mean() if len(sub_t1) > 0 else np.nan
        out[f'{c}_t2'] = sub_t2[c].mean() if len(sub_t2) > 0 else np.nan
    return pd.Series(out)

HW_WINDOW_COLS = [f'{c}_t{t}' for t in [1, 2] for c in HW_METRIC_COLS]

print('Computing HW window stats for train/test regions ...')
hw_feat_split = split.apply(hw_window_stats, axis=1, t3_col='t3')
# Zero-fill regions where no HW events were detected (e.g. Genemuiden/IJssel2)
no_events_mask = hw_feat_split[HW_WINDOW_COLS].isna().all(axis=1)
hw_feat_split.loc[no_events_mask, HW_WINDOW_COLS] = 0
split[HW_WINDOW_COLS] = hw_feat_split[HW_WINDOW_COLS]

print('Computing HW window stats for inference regions (t1 window only) ...')
hw_feat_inf = inference.apply(hw_window_stats, axis=1, t3_col=None)
no_events_mask_inf = hw_feat_inf[HW_WINDOW_COLS].isna().all(axis=1)
hw_feat_inf.loc[no_events_mask_inf, HW_WINDOW_COLS] = 0
# t2 window undefined for 2-timestamp — leave as NaN (t3 unknown)
inference[HW_WINDOW_COLS] = hw_feat_inf[HW_WINDOW_COLS]

n_ok = split[HW_WINDOW_COLS].notna().all(axis=1).sum()
print(f'  HW assigned (split): {n_ok:,} / {len(split):,}')

## 8. Bend exposure (curvature features)

Curvature features (`bend_exposure_n5`, `bend_exposure_n8`) were computed by `src/data/curvature.py`
and stored in `region_features_v2.parquet`. We load them from there rather than recomputing
(the curvature computation depends on scope geometries ordered by chainage which is non-trivial to replicate here).

In [ ]:
CURV_KEEP = ['bend_exposure_n5', 'bend_exposure_n8']
print('Loading bend exposure from reference features_v2 ...')
feat_v2 = pd.read_parquet(REFERENCE_FEATURES_V2, columns=CURV_KEEP)

split[CURV_KEEP]     = feat_v2.reindex(split.index)[CURV_KEEP]
inference[CURV_KEEP] = feat_v2.reindex(inference.index)[CURV_KEEP]
n_ok = split[CURV_KEEP].notna().all(axis=1).sum()
print(f'  Assigned: {n_ok:,} / {len(split):,} split regions')
print(f'  Missing (inference): {inference[CURV_KEEP].isna().any(axis=1).sum():,}')

## 8b. Erosion volume rate

`erosion_vol_rate_t1` is computed in `02_region_split.ipynb` from `erosion_vlakken_filtered`.
For inference_only (2-timestamp) locations, impute with 0 (the mode: 73% of OK regions have 0).

In [ ]:
# For split: rename erosion_vol_train_rate → erosion_vol_rate_t1 if present
if 'erosion_vol_train_rate' in split.columns:
    split['erosion_vol_rate_t1'] = split['erosion_vol_train_rate']
elif 'erosion_vol_rate_t1' not in split.columns:
    print('WARNING: erosion_vol_rate_t1 not found in region_split — will be NaN')
    split['erosion_vol_rate_t1'] = np.nan

# For inference: impute with 0 (mode for 3-timestamp regions)
inference['erosion_vol_rate_t1'] = 0.0
print('erosion_vol_rate_t1 summary (split):')
print(split['erosion_vol_rate_t1'].describe().round(3).to_string())
print(f'\nInference locations: imputed to 0 ({len(inference):,} regions)')

## 9. Ordinal encoding

In [ ]:
CAT_COLS = {
    'river':             'river',
    'vegetation_class':  'rws_vegetatielegger:vegetatieklassen_majority_class_vlklasse',
    'land_use':          'BrpGewas_majority_class_category',
    'soil_group':        'soil_group',
}
for df in [split, inference]:
    for col, key in CAT_COLS.items():
        df[f'{col}_enc'] = encode_col(df[col], key)

print('Encoding check (unknown → -1):')
for col in ['river_enc', 'vegetation_class_enc', 'land_use_enc', 'soil_group_enc']:
    unk = (split[col] == CONST.DEFAULT_UNKNOWN_CATEGORY_LABEL).sum()
    print(f'  {col}: {unk:,} unknown  ({unk/len(split):.1%})')

## 10. Assemble output tables

In [ ]:
COLS_OUT = [
    'v_train', 'v_test',
    'dist_t1', 'dist_t2', 'dist_t3',
    'train_span_yr', 'test_span_yr',
    'is_nvo',
    'river', 'river_enc',
    'vegetation_class', 'vegetation_class_enc',
    'land_use', 'land_use_enc',
    'erosion_vol_rate_t1',
    'soil_group', 'soil_group_enc',
    'n_events_t1', 'max_rise_rate_t1', 'drawdown_index_t1', 'flood_days_t1',
    'n_events_t2', 'max_rise_rate_t2', 'drawdown_index_t2', 'flood_days_t2',
    'bend_exposure_n5', 'bend_exposure_n8',
    'split', 'cluster',
]

# Train/test: all 29 columns
features_out = split[[c for c in COLS_OUT if c in split.columns]]
# Inference: same schema minus v_test and split (undefined for 2-timestamp)
inf_cols_out = [c for c in COLS_OUT if c not in ('v_test', 'split') and c in inference.columns]
inference_out = inference[inf_cols_out]

print(f'region_features shape       : {features_out.shape}')
print(f'region_inference_features   : {inference_out.shape}')
print(f'Columns: {list(features_out.columns)}')

## 11. Parity check — CONTROL EXPERIMENT

Before saving, assert that the new `region_features` matches the reference
`02_processed/erosion/region_features.parquet` for all shared 7,444 OK locations.

**This is the gate condition for proceeding to notebook 04.**

In [ ]:
ref = pd.read_parquet(REFERENCE_FEATURES)
shared = features_out.index.intersection(ref.index)
print(f'Shared locations: {len(shared):,} (new) vs {len(ref):,} (reference)')

# ── Schema check ──────────────────────────────────────────────────────────────
new_cols = set(features_out.columns)
ref_cols = set(ref.columns)
extra_new = new_cols - ref_cols
missing_new = ref_cols - new_cols
if extra_new:   print(f'  Extra cols in new    : {sorted(extra_new)}')
if missing_new: print(f'  Missing from new     : {sorted(missing_new)}')
assert not missing_new, f'New features are missing columns present in reference: {missing_new}'
print('  Column schema: OK')

# ── Value check for numeric columns ──────────────────────────────────────────
numeric_cols = features_out.select_dtypes(include='number').columns.tolist()
# Exclude split assignment (may differ by design if random seed unchanged, but check it)
TOLERANCE = 1e-6
fails = []
for col in numeric_cols:
    if col not in ref.columns: continue
    new_vals = features_out.loc[shared, col].values
    ref_vals = ref.loc[shared, col].values
    # Compare non-null entries
    mask = ~(np.isnan(new_vals) | np.isnan(ref_vals))
    if mask.sum() == 0: continue
    max_diff = np.abs(new_vals[mask] - ref_vals[mask]).max()
    if max_diff > TOLERANCE:
        fails.append((col, max_diff))
        print(f'  FAIL: {col}  max_diff={max_diff:.6f}')

if not fails:
    print(f'  Numeric values: OK (all {len(numeric_cols)} numeric columns within tolerance={TOLERANCE})')
else:
    print(f'\n  {len(fails)} columns differ beyond tolerance. Investigate before proceeding.')

# ── Categorical check ─────────────────────────────────────────────────────────
cat_cols_check = ['vegetation_class', 'land_use', 'soil_group', 'river']
cat_fails = []
for col in cat_cols_check:
    if col not in ref.columns: continue
    mismatch = (features_out.loc[shared, col] != ref.loc[shared, col]).sum()
    if mismatch > 0:
        cat_fails.append((col, mismatch))
        print(f'  FAIL: {col}  mismatched={mismatch}')
if not cat_fails:
    print(f'  Categorical values: OK')

if not fails and not cat_fails:
    print('\n✓ PARITY CHECK PASSED — proceeding to save and notebook 04 is unblocked.')
else:
    print('\n✗ PARITY CHECK FAILED — do not proceed to notebook 04 without resolving diffs.')

## 12. Save

In [ ]:
features_out.to_parquet(OUT_FEATURES)
inference_out.to_parquet(OUT_INFERENCE_FEATURES)

print(f'Saved region_features            → {OUT_FEATURES}')
print(f'  Shape: {features_out.shape}')
print(f'Saved region_inference_features  → {OUT_INFERENCE_FEATURES}')
print(f'  Shape: {inference_out.shape}')
print(f'\nNull counts (region_features):')
print(features_out.isnull().sum()[features_out.isnull().sum() > 0].to_string())